In [ ]:
# imports
import pandas as pd
import numpy as np
import anndata as ad
import anndata as ad, yaml
import sys, os

# variables
path = "../../data/GTEx/GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_reads.parquet"

# functions
def load_ncbi_gene_ids()->pd.DataFrame:
    yml_path = "/aloy/home/ddalton/projects/disease_signatures/conf/paths.yml"
    # Load YAML content
    with open(yml_path, "r") as f:
        config = yaml.safe_load(f)
    database_dir = config["database_dir"]
    return pd.read_csv(os.path.join(database_dir, "NCBI/gene_info"), sep="\t", usecols=["#tax_id", "GeneID", "type_of_gene", "Symbol"]
        )

# load GTEx
# GEx
df = pd.read_parquet(path)          # usually genes x samples

# mappings 
attrs = pd.read_csv(
    "../../data/GTEx/GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt",
    sep="\t"
)
gtexid_2_tiss = dict(zip(attrs["SAMPID"],attrs["SMTS"]))
gtexid_2_tiss_spec = dict(zip(attrs["SAMPID"],attrs["SMTSD"]))


# load mapping to human protein coding genes
ncbi = load_ncbi_gene_ids()
ncbi = ncbi[ncbi["#tax_id"] == 9606]
ncbi = ncbi[ncbi["type_of_gene"] == "protein-coding"]

coding_symbols = set(ncbi["Symbol"])
print("protein coding symbols:", len(coding_symbols))

# substet to protein coding genes
df_pc = df.query("Description in @coding_symbols")
print(f"Filtered to protein coding genes {df_pc.shape}")

In [ ]:
df_pc.columns


Index(['Description', 'GTEX-1117F-0005-SM-HL9SH',
       'GTEX-1117F-0011-R10b-SM-GI4VE', 'GTEX-1117F-0011-R11b-SM-GIN8R',
       'GTEX-1117F-0011-R2b-SM-GI4VL', 'GTEX-1117F-0011-R3a-SM-GJ3PJ',
       'GTEX-1117F-0011-R4b-SM-GI4VM', 'GTEX-1117F-0011-R5a-SM-GI4VW',
       'GTEX-1117F-0011-R6a-SM-GI4VX', 'GTEX-1117F-0011-R7a-SM-H65ZK',
       ...
       'GTEX-ZZPU-1326-SM-5GZWS', 'GTEX-ZZPU-1426-SM-5GZZ6',
       'GTEX-ZZPU-1826-SM-5E43L', 'GTEX-ZZPU-2126-SM-5EGIU',
       'GTEX-ZZPU-2226-SM-5EGIV', 'GTEX-ZZPU-2326-SM-GOQYU',
       'GTEX-ZZPU-2426-SM-5E44I', 'GTEX-ZZPU-2526-SM-GOQZ3',
       'GTEX-ZZPU-2626-SM-5E45Y', 'GTEX-ZZPU-2726-SM-5NQ8O'],
      dtype='object', length=19789)

In [22]:
# count matrix
count_mat = df_pc.drop(columns=["Description"])
X = count_mat.to_numpy(dtype=np.float32).T

# gene symbols
gene_symbols = df_pc["Description"].astype(str).to_numpy()

# create adata file
adata = ad.AnnData(X)
adata.obs_names = count_mat.columns.astype(str)   # sample IDs
adata.var_names = gene_symbols                    # gene symbols

# store tissues in obs (aligned to obs_names)
adata.obs["tissue"] = attrs.reindex(adata.obs_names)["SMTS"].values
adata.obs["tissue_detail"] = attrs.reindex(adata.obs_names)["SMTSD"].values



# keep raw counts in layers
adata.layers["X"] = adata.X.copy()